# Alert 생성기 실험 ⑥ — 룰이 고른 거래를 생성기에 넣으면

씨앗만 바꿔서 **같은 생성기**를 세 번 돌린다. 05번은 룰 적중을 그대로 Alert 로 만들었고, 여기서는 룰이 고른 거래를 생성기가 묶는다.

| 씨앗 | 무엇인가 |
|---|---|
| M 모델 | 01번의 씨앗 (모델 씨앗 + 씨앗 보강). 지금 서비스 방식 |
| R 룰 | 03번에서 룰 6종이 고른 거래. 모델을 전혀 쓰지 않는다 |
| MR 모델 + 룰 | 둘의 합집합. 서비스에 룰 씨앗을 얹는 방식 (룰엔진 설계안의 "Alert 생성") |

- 01번, 03번을 같은 DATASET 으로 먼저 돌려야 한다. 05번을 돌려 두었으면 그 결과(룰 적중 그대로 = R2)를 표에 같이 놓는다.
- 생성기는 01번과 같이 서비스 저장소(`money_laundry`)의 코드를 그대로 불러온다. 정책도 기본값 그대로다.
- 씨앗 보강은 모델 점수로 하는 것이라 M 에만 들어 있다. R 에는 보강이 없고, MR 은 M 의 씨앗에 룰 씨앗을 더한 것이다.
- 정답 · 제외 패턴 · "묶음은 거래 2건 이상" 규칙과 지표는 02번 · 05번과 같다.

## 0. 설정

In [ ]:
DATASET = 'HI-Medium'            # 01번과 03번을 같은 DATASET 으로 먼저 돌려야 한다
GEN_RUN = 'before_after_grouping'        # 01번 결과 폴더 (members.parquet: 모델 씨앗)
RULE_RUN = 'rule_vs_model'               # 03번 결과 폴더 (hits.parquet: 룰이 고른 거래)
ALERT_RUN = 'rule_alerts'                # 05번 결과 폴더 (있으면 표에 같이 놓는다)
EXCLUDE = ['STACK', 'BIPARTITE']         # 평가에서 빼는 패턴 (02번과 같다)
BUDGET_SCALE = 1                 # 생성기 탐색 예산 배수. 3절에서 예산 초과가 나면 올린다

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json, subprocess, time
import duckdb, numpy as np, pandas as pd
import gen_lib as GL
import rule_lib as RL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir() or (p / 'datasets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
GEN_DIR = Path.cwd() / 'results' / DATASET / GEN_RUN
RULE_DIR = Path.cwd() / 'results' / DATASET / RULE_RUN
OUT = Path.cwd() / 'results' / DATASET / 'rule_seeds_generator'
OUT.mkdir(parents=True, exist_ok=True)
gen_run = json.load(open(GEN_DIR / 'run.json', encoding='utf-8'))
rule_run = json.load(open(RULE_DIR / 'run.json', encoding='utf-8'))
assert rule_run['test_window'] == gen_run['test_window'], '03번과 01번의 평가 구간이 다르다'
a, b = gen_run['test_window']
PAD_DAYS = gen_run['pad_days']          # 평가 구간 앞으로 더 읽는 날짜 수. 01번과 같게 둔다
print('평가 구간', a, '~', b, '| 앞으로 더 읽는 날', PAD_DAYS, '| 01번 생성기', gen_run['generator_commit'])

## 1. 생성기 불러오기

In [ ]:
WORKER = GL.find_worker()
G = GL.load_generator(WORKER)
POLICY = G.BlockPolicy()
COMMIT = subprocess.run(['git', '-C', str(WORKER), 'log', '-1', '--format=%h %ad %s', '--date=short'], capture_output=True, text=True).stdout.strip()
print(WORKER)
print('정책', POLICY.version, '|', COMMIT)

## 2. 거래 읽기
01번과 같다. 평가 구간과 그 앞 `PAD_DAYS` 일을 시간순으로 읽고, 보낸 금액을 USD 로 바꾸고, 패턴 라벨을 붙인다.

In [ ]:
con = duckdb.connect()
first_day = (pd.Timestamp(a) - pd.Timedelta(days=PAD_DAYS)).date()
t = time.perf_counter()
tx = con.execute(f"""
    SELECT edge_id, ts, src, dst, paid, payment_currency, CAST(is_laundering AS TINYINT) AS y, day
    FROM read_parquet('{BASE / 'transactions.parquet'}')
    WHERE day BETWEEN DATE '{first_day}' AND DATE '{b}'
    ORDER BY ts, edge_id""").df()
tx['usd'] = GL.to_usd(tx.paid, tx.payment_currency)
pat = con.execute(f"""SELECT edge_id, any_value(pattern_type) AS pattern_type
                      FROM read_parquet('{BASE / 'patterns.parquet'}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
n_before = len(tx)
tx = tx.merge(pat, on='edge_id', how='left')
assert len(tx) == n_before and tx.ts.is_monotonic_increasing        # 행 수와 순서가 그대로여야 한다
tx['pattern_type'] = np.where(tx.y == 1, tx.pattern_type.fillna('NO_PATTERN'), '')
y, pattern_type = tx.y.to_numpy(), tx.pattern_type.to_numpy()
in_eval = ((pd.to_datetime(tx.day) >= pd.Timestamp(a)) & (pd.to_datetime(tx.day) <= pd.Timestamp(b))).to_numpy()
pattern_mask = (y == 1) & (pattern_type != 'NO_PATTERN')
print(f'{len(tx):,} 건 ({time.perf_counter() - t:.0f}s) | 평가 구간 거래 {in_eval.sum():,} | 그중 세탁 {int(y[in_eval].sum()):,} | 환율 모르는 거래 {int(tx.usd.isna().sum())}')

## 3. 씨앗 세 가지
- 모델 쪽 씨앗은 01번의 `members.parquet` 에서 읽는다(모델 씨앗 + 보강으로 들어온 씨앗). 점수 파일을 다시 읽지 않는다.
- 룰 쪽 씨앗은 03번의 `hits.parquet` 에서 룰 6종 중 하나라도 고른 평가 구간 거래다.

In [ ]:
members = pd.read_parquet(GEN_DIR / 'members.parquet')
hits = pd.read_parquet(RULE_DIR / 'hits.parquet')
RULE_NAMES = [name for name in RL.RULES.values() if name in hits.columns]
row_of = pd.Series(np.arange(len(tx)), index=tx.edge_id.to_numpy())          # edge_id → 행 번호

def rows_mask(edge_ids):
    mask = np.zeros(len(tx), dtype=bool)
    mask[row_of.reindex(edge_ids).dropna().astype(int).to_numpy()] = True
    return mask

seed_m = rows_mask(members.edge_id[members.seed_model | members.promoted])
seed_r = rows_mask(hits.edge_id[hits[RULE_NAMES].any(axis=1)]) & in_eval
assert int(seed_m.sum()) == int((members.seed_model | members.promoted).sum()) and not (seed_m & ~in_eval).any()
SEEDS = {'M 모델': seed_m, 'R 룰': seed_r, 'MR 모델 + 룰': seed_m | seed_r}
stages = [GL.stage_metrics(f'씨앗 · {name}', mask, y, in_eval, pattern_mask) for name, mask in SEEDS.items()]
pd.DataFrame(stages)

In [ ]:
# 두 씨앗이 얼마나 겹치나
parts = {'둘 다': seed_m & seed_r, '모델만': seed_m & ~seed_r, '룰만': seed_r & ~seed_m}
seed_overlap = pd.DataFrame([dict(씨앗=name, 거래=int(mask.sum()), 세탁=int(y[mask].sum()),
                                  세탁_비율=round(float(y[mask].mean()), 4) if mask.any() else None) for name, mask in parts.items()]).set_index('씨앗')
seed_overlap

## 4. 생성기 실행
씨앗마다 한 번씩, 한 번에 넣고 묶는다. 거래 색인을 매번 새로 만들어서 HI-Medium 은 한 번에 1 ~ 2분 걸린다.

In [ ]:
t = time.perf_counter()
edges = GL.make_edges(G, tx)
print(f'생성기 입력 {len(edges):,} 건 ({time.perf_counter() - t:.0f}s)')
runs = {}
for name, seeds in SEEDS.items():
    try:
        batch, info = GL.run_generator(G, edges, np.flatnonzero(seeds), budget_scale=BUDGET_SCALE)
    except G.GraphBudgetExceeded as error:
        raise RuntimeError(f'생성기 탐색 예산 초과({name}): {error}. 0절의 BUDGET_SCALE 을 올려 다시 돌린다') from error
    role, graph, reason, alerts = GL.batch_table(batch, len(tx))
    runs[name] = SimpleNamespace(role=role, graph=graph, reason=reason, alerts=GL.alert_purity(graph, role, y, alerts), info=info)
    del batch
pd.DataFrame({name: run.info for name, run in runs.items()})

In [ ]:
# 확인: M 은 01번과 같은 씨앗 · 같은 생성기라서 Alert 구성이 01번과 같아야 한다 (생성기 커밋이 다르면 달라질 수 있다)
old = members[members.role > 0].set_index('edge_id').alert.rename('old')
new = pd.Series(runs['M 모델'].graph, index=tx.edge_id.to_numpy(), name='new')
new = new[new >= 0]
both = pd.concat([old, new], axis=1)
same = both.notna().all(axis=None) and both.groupby('old').new.nunique().max() == 1 and both.groupby('new').old.nunique().max() == 1
print('01번과 같다' if same else f'01번과 다르다: 01번 Alert 거래 {len(old):,}, 지금 {len(new):,}, 한쪽에만 있는 거래 {int(both.isna().any(axis=1).sum()):,} (01번 커밋 {gen_run["generator_commit"]} / 지금 {COMMIT})')

## 5. 거래 단위
씨앗이 생성기를 거친 뒤 Alert 에 남은 거래(씨앗 + 연결 거래)를 평가 구간 안에서 센다.

In [ ]:
stages += [GL.stage_metrics(f'묶은 뒤 · {name}', run.role > GL.ROLE_NONE, y, in_eval, pattern_mask) for name, run in runs.items()]
stage_table = pd.DataFrame(stages)
stage_table

In [ ]:
# Alert 에 못 든 씨앗과, 씨앗이 아닌데 들어온 거래(연결 거래)
rows = []
for name, run in runs.items():
    dropped = SEEDS[name] & (run.role == GL.ROLE_NONE)
    linked = run.role == GL.ROLE_CONNECTION
    rows.append(dict(씨앗=name, Alert=len(run.alerts), 씨앗_수=int(SEEDS[name].sum()), 못_든_씨앗=int(dropped.sum()), 못_든_씨앗_중_세탁=int(y[dropped].sum()),
                     연결_거래=int(linked.sum()), 연결_거래_중_세탁=int(y[linked].sum())))
flow = pd.DataFrame(rows).set_index('씨앗')
flow

## 6. 정답(세탁 시도)과 비교
02번 · 05번과 같은 방법이다. 시도마다 평가 구간 안의 거래가 한 Alert 에 얼마나 온전히 들어갔는지 본다.

In [ ]:
start, end = pd.Timestamp(a), pd.Timestamp(b) + pd.Timedelta(days=1)
PAT = BASE / 'patterns.parquet'
IN = f"ts >= TIMESTAMP '{start}' AND ts < TIMESTAMP '{end}'"
attempts = con.execute(f"""SELECT attempt_id, any_value(pattern_type) AS pattern_type, count(*) AS n_rows,
                                  count(*) FILTER (WHERE {IN}) AS n_in, count(edge_id) FILTER (WHERE {IN}) AS n_in_matched,
                                  min(ts) AS first_ts, max(ts) AS last_ts
                           FROM read_parquet('{PAT}') GROUP BY attempt_id""").df()
attempts['eligible'] = (~attempts.pattern_type.isin(EXCLUDE) & (attempts.n_in >= 2) & (attempts.n_in == attempts.n_in_matched))
attempt_of = con.execute(f"""SELECT edge_id, min(attempt_id) AS attempt_id, arg_min(pattern_type, attempt_id) AS pattern_type,
                                    bool_and({IN}) AS in_window
                             FROM read_parquet('{PAT}') WHERE edge_id IS NOT NULL GROUP BY edge_id""").df()
attempt_of = attempt_of[~attempt_of.pattern_type.isin(EXCLUDE)]
eval_attempts = set(attempts.loc[attempts.eligible, 'attempt_id'])
truth = attempt_of[attempt_of.attempt_id.isin(eval_attempts) & attempt_of.in_window]      # 정답: 평가할 시도의 구간 안 거래
print(f'평가할 시도 {len(eval_attempts):,} 개 | 정답 거래 {len(truth):,} 건')

In [ ]:
# 묶기 전의 상한: 시도의 거래 중 씨앗이 몇 건인가
rows = []
for name, mask in SEEDS.items():
    ids = set(tx.edge_id.to_numpy()[mask])
    s = truth.assign(hit=truth.edge_id.isin(ids)).groupby('attempt_id').agg(size=('hit', 'size'), hit=('hit', 'sum'))
    rows.append(dict(씨앗=name, 시도_수=len(s), 정답_거래=int(s['size'].sum()), 씨앗인_거래=int(s.hit.sum()),
                     거래_재현율=round(float(s.hit.sum() / s['size'].sum()), 4), 전부_씨앗인_시도=int((s.hit == s['size']).sum()),
                     두_건_이상_씨앗인_시도=int((s.hit >= 2).sum()), 씨앗이_없는_시도=int((s.hit == 0).sum())))
ceiling = pd.DataFrame(rows).set_index('씨앗')
ceiling

In [ ]:
def as_groups(run):          # 02번과 같다: Alert 구성 거래에서 제외 패턴을 빼고, 거래 2건 이상인 Alert 만
    member = (run.role > GL.ROLE_NONE) & ~np.isin(pattern_type, EXCLUDE)
    g = pd.DataFrame(dict(edge_id=tx.edge_id.to_numpy()[member], group=run.graph[member], y=y[member]))
    return g[g.groupby('group')['edge_id'].transform('size') >= 2]

groups = {name: as_groups(run) for name, run in runs.items()}
summaries, per_attempt = [], {}
for name, g in groups.items():
    summary, per = RL.compare_alerts(name, g, attempt_of, eval_attempts, truth)
    summaries.append(summary)
    per_attempt[name] = per
result = pd.DataFrame(summaries).set_index('방법').T.astype(object)
other = Path.cwd() / 'results' / DATASET / ALERT_RUN / 'summary.csv'       # 05번: 룰 적중을 그대로 Alert 로 (생성기 없음)
if other.exists():
    result['룰 적중 그대로 (05번 R2)'] = pd.read_csv(other, index_col=0)['R2 겹치면 합침'].reindex(result.index)
result

**읽는 법** (02번 · 05번과 같은 항목)
- `세탁_비율`: Alert 에 든 거래 중 세탁 거래의 비율.
- `시도_2개이상_섞인_묶음`: 서로 다른 세탁 시도가 한 Alert 에 들어간 수.
- `쌍_정밀도`: 같은 Alert 에 든 두 거래가 실제로 같은 시도일 비율. `쌍_재현율`: 같은 시도의 두 거래가 같은 Alert 에 들어갔을 비율.
- `온전히_한_묶음`: 시도의 (구간 안) 거래 전부가 한 Alert 에 들어간 시도 수. 위 표의 `전부_씨앗인_시도` 가 대략의 상한이다(연결 거래로 되찾는 경우 제외).
- `쪼개진_시도`: 시도의 거래가 Alert 2개 이상에 나뉜 수. `전혀_안_묶인_시도`: 어느 Alert 에도 안 든 수.
- R 과 `룰 적중 그대로` 는 **씨앗이 같고 묶는 방법만 다르다.** 둘의 차이가 생성기가 한 일이다.

## 6-1. 패턴별 · 시도의 끝별

In [ ]:
def table_by(keys, per_map):
    frames = []
    for name, per in per_map.items():
        f = per.groupby(keys, observed=True).agg(시도=('size', 'size'), 온전=('coverage', lambda s: round(float((s == 1).mean()), 3)),
                                                 포함률=('coverage', lambda s: round(float(s.mean()), 3)),
                                                 쪼개짐=('n_groups', lambda s: round(float((s >= 2).mean()), 3)),
                                                 안_묶임=('covered', lambda s: round(float((s == 0).mean()), 3)))
        frames.append(f.add_prefix(name.split()[0] + '_'))
    return pd.concat(frames, axis=1)

by_pattern = table_by('pattern_type', per_attempt)
by_pattern

In [ ]:
after = attempts.set_index('attempt_id').last_ts >= end          # 시도 → 평가 구간 뒤로 이어지는가
by_tail = table_by('끝', {name: per.assign(끝=per.attempt_id.map(after).map(GL.TAIL_LABELS)) for name, per in per_attempt.items()})
by_tail

## 7. 모델에 룰을 얹으면 무엇이 달라지나 (M → MR)
### 7-1. 룰만 고른 씨앗은 어디로 갔나

In [ ]:
m, mr = runs['M 모델'], runs['MR 모델 + 룰']
member_mr = mr.role > GL.ROLE_NONE
model_seeds_in = pd.Series(seed_m[member_mr]).groupby(mr.graph[member_mr]).sum()          # MR 의 Alert 마다 모델 쪽 씨앗 수
with_model = np.zeros(len(tx), dtype=bool)
with_model[member_mr] = model_seeds_in.reindex(mr.graph[member_mr]).to_numpy() > 0         # 모델 쪽 씨앗이 있는 Alert 에 든 거래
only_rule = seed_r & ~seed_m
where = np.select([~member_mr, with_model], ['Alert 에 못 듦', '모델 씨앗이 있는 Alert 에 붙음'], '모델 씨앗이 없는 새 Alert')
fate = (pd.DataFrame(dict(간_곳=where[only_rule], y=y[only_rule])).groupby('간_곳').agg(거래=('y', 'size'), 세탁=('y', 'sum')))
fate['세탁_비율'] = (fate.세탁 / fate.거래).round(4)
fate

In [ ]:
# Alert 에 못 든 룰 씨앗의 탈락 사유
pd.crosstab(pd.Series(mr.reason[only_rule & ~member_mr], name='탈락 사유'), pd.Series(np.where(y[only_rule & ~member_mr] == 1, '세탁', '정상'), name='실제'), margins=True)

### 7-2. 룰 때문에 새로 생긴 Alert
모델 쪽 씨앗이 하나도 없는 Alert 는 룰이 없었으면 생기지 않았을 Alert 다.

In [ ]:
stat = pd.DataFrame(dict(alert=mr.graph[member_mr], y=y[member_mr], 모델_씨앗=seed_m[member_mr], in_eval=in_eval[member_mr]))
per_alert = stat.groupby('alert').agg(거래=('y', 'size'), 세탁=('y', 'sum'), 모델_씨앗=('모델_씨앗', 'sum'))
per_alert['구성'] = np.where(per_alert.모델_씨앗 == 0, '모델 씨앗 없음 (룰 때문에 생김)', '모델 씨앗 있음')
new_alerts = per_alert.groupby('구성').agg(Alert=('거래', 'size'), 거래=('거래', 'sum'), 세탁=('세탁', 'sum'),
                                           세탁_0건_Alert=('세탁', lambda s: int((s == 0).sum())), 전부_세탁_Alert=('세탁', lambda s: int((s == per_alert.loc[s.index, '거래']).sum())))
new_alerts['세탁_비율'] = (new_alerts.세탁 / new_alerts.거래).round(4)
new_alerts

### 7-3. 원래 있던 Alert 는 어떻게 됐나
M 의 Alert 하나하나가 MR 에서 어떻게 됐는지 센다.
- `그대로`: 구성 거래가 같다 / `커짐`: 같은 Alert 인데 거래가 늘었다
- `다른 Alert 와 합쳐짐`: M 의 Alert 둘 이상이 MR 에서 한 Alert 가 됐다
- `나뉨 · 줄어듦`: 구성 거래가 MR 의 Alert 여러 개로 갈라졌거나 일부가 Alert 밖으로 나갔다

In [ ]:
member_m = m.role > GL.ROLE_NONE
pair = pd.DataFrame(dict(old=m.graph[member_m], new=mr.graph[member_m]))                  # M 의 구성 거래가 MR 에서 든 Alert (-1 = 밖)
n_new = pair.groupby('old').new.nunique()                                                 # M Alert 가 MR 에서 몇 군데로 갔나
n_old = pair[pair.new >= 0].groupby('new').old.nunique()                                  # MR Alert 가 M Alert 몇 개를 담았나
target = pair.groupby('old').new.first()
size_old = pair.groupby('old').size()
size_new = pd.Series(mr.graph[member_mr]).value_counts()
kind = pd.Series('나뉨 · 줄어듦', index=n_new.index)
one = (n_new == 1) & (target >= 0)
kind[one & (target.map(n_old) >= 2)] = '다른 Alert 와 합쳐짐'
alone = one & (target.map(n_old) == 1)
kind[alone & (target.map(size_new) == size_old)] = '그대로'
kind[alone & (target.map(size_new) > size_old)] = '커짐'
alert_change = kind.value_counts().rename('M 의 Alert 수').to_frame()
alert_change['비율'] = (alert_change['M 의 Alert 수'] / len(kind)).round(4)
alert_change

## 8. 룰만 넣었을 때 (R)
생성기가 룰 씨앗을 어떻게 묶었는지 본다. 묶은 구조 근거(`kinds`)가 룰의 패턴과 맞는지도 같이 보인다.

In [ ]:
r = runs['R 룰']
dropped = seed_r & (r.role == GL.ROLE_NONE)
pd.crosstab(pd.Series(r.reason[dropped], name='탈락 사유'), pd.Series(np.where(y[dropped] == 1, pattern_type[dropped], '정상'), name='실제'), margins=True)

In [ ]:
# 어떤 구조 근거로 묶였나 (STAR_IN, STAR_OUT, RELAY, PASS, PATH, BRIDGE, ...). 위 10개
kinds = r.alerts.groupby('kinds').agg(Alert=('alert', 'size'), 세탁_비율_평균=('purity', 'mean'), 거래_평균=('members', 'mean')).sort_values('Alert', ascending=False).round(3)
kinds.head(10)

## 9. 저장

In [ ]:
result.to_csv(OUT / 'summary.csv')
stage_table.to_csv(OUT / 'stages.csv', index=False)
seed_overlap.to_csv(OUT / 'seed_overlap.csv'); flow.to_csv(OUT / 'flow.csv'); ceiling.to_csv(OUT / 'ceiling.csv')
by_pattern.to_csv(OUT / 'by_pattern.csv'); by_tail.to_csv(OUT / 'by_tail.csv')
fate.to_csv(OUT / 'rule_seed_fate.csv'); new_alerts.to_csv(OUT / 'new_alerts.csv'); alert_change.to_csv(OUT / 'alert_change.csv')
kinds.to_csv(OUT / 'kinds_rule_only.csv')
pd.concat([per.assign(method=name) for name, per in per_attempt.items()]).to_csv(OUT / 'per_attempt.csv', index=False)
keep = (seed_m | seed_r) | np.logical_or.reduce([run.role > GL.ROLE_NONE for run in runs.values()])
out = dict(edge_id=tx.edge_id[keep], y=y[keep], pattern_type=pattern_type[keep], seed_model=seed_m[keep], seed_rule=seed_r[keep])
for name, run in runs.items():
    key = name.split()[0]
    out[f'role_{key}'], out[f'alert_{key}'], out[f'reason_{key}'] = run.role[keep], run.graph[keep], run.reason[keep]
pd.DataFrame(out).to_parquet(OUT / 'members.parquet', index=False)
json.dump(dict(dataset=DATASET, gen_run=GEN_RUN, rule_run=RULE_RUN, test_window=[a, b], pad_days=PAD_DAYS, exclude=EXCLUDE,
               generator_commit=COMMIT, same_as_01=bool(same), info={name: run.info for name, run in runs.items()}, chosen=rule_run['chosen']),
          open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(sorted(p.name for p in OUT.iterdir()))